# Agentic Testing — Final Experiment & Two-LLM Comparison

This notebook is the final reproducible entry point for the merged Member 2 + Member 3 + Member 4 project.

## Systems
- **Baseline A** — single-shot generation
- **Baseline B** — one-shot multi-agent consensus
- **Variant 1** — mutant-guided iterative refinement with error-trace feedback
- **Variant 2** — mutant-guided iterative refinement with state-prediction prompting

## Required two-LLM comparison
Before the full experiment, the notebook makes **the same single Baseline-A-style generation call against the same function and the same prompt with two different LLMs**:
- Gemini
- Groq

Both generated suites are then scored with the same mutation/coverage/pytest scorer. This gives a direct, controlled LLM comparison without mixing the two models inside a system condition.

The two-LLM comparison is a **sanity/comparison experiment**, not a replacement for the locked 3-repeat paper experiment. Its outputs are stored separately under `logs/llm_comparison/`.

> **API keys are entered at runtime and are never written into the repository.**


In [ ]:
# 1. Locate the repository.
from pathlib import Path
import os, sys, subprocess, json, textwrap, shutil, time

candidates = [
    Path.cwd(),
    Path("/content/AI-Agentic-Testing"),
    Path("/content/Agentic-AI-Testing-main"),
    Path("/content/Agentic-AI-Testing-main/Agentic-AI-Testing-main"),
]
REPO = next((p for p in candidates if (p / "baselines").exists() and (p / "refinement_loop").exists()), None)
if REPO is None:
    raise FileNotFoundError(
        "Repository not found. In Colab, clone the GitHub repository or upload/extract "
        "the finalized project so a directory containing baselines/, evaluation/, and refinement_loop/ exists."
    )

os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("Repository:", REPO)


In [ ]:
# 2. Install the pinned project dependencies.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
print("Dependencies installed.")


In [ ]:
# 3. Offline acceptance tests — do this before spending any API quota.
r = subprocess.run(
    [sys.executable, "-m", "pytest", "refinement_loop/tests/", "baselines/smoke_test.py", "-q"],
    text=True, capture_output=True
)
print(r.stdout)
if r.returncode != 0:
    print(r.stderr)
    raise RuntimeError("Offline acceptance tests failed.")
print("Offline acceptance tests passed.")


## Configure both LLMs

The next cell requests **both** API keys. The keys are kept only in the current Colab/Python process.

For the direct comparison, both models receive the **same function source, same system prompt, and same user prompt**, and each model gets exactly one generation call.

For the full experiment, choose which provider should run the expensive 30-function × 3-repeat experiment. The two-LLM comparison is always performed first.


In [ ]:
# 4. Enter both live API keys.
import getpass, os

gemini_key = getpass.getpass("GEMINI_API_KEY: ").strip()
groq_key = getpass.getpass("GROQ_API_KEY: ").strip()

if not gemini_key:
    raise ValueError("GEMINI_API_KEY is required.")
if not groq_key:
    raise ValueError("GROQ_API_KEY is required.")

os.environ["GEMINI_API_KEY"] = gemini_key
os.environ["GROQ_API_KEY"] = groq_key

groq_model = input(
    "GROQ_MODEL_ID (default: llama-3.3-70b-versatile): "
).strip() or "llama-3.3-70b-versatile"
os.environ["GROQ_MODEL_ID"] = groq_model

print("Both API keys configured in memory.")
print("Groq model:", groq_model)


## Direct two-LLM, one-call comparison

This is the requested minimal comparison:

**one identical generation task → Gemini once + Groq once → same evaluator**

It is intentionally isolated from the main `logs/*.json` experiment records so it cannot contaminate the paper's 3-repeat system comparison.


In [ ]:
# 5. One-call head-to-head comparison on the SAME function and SAME prompt.
from baselines import baseline_a, config as baselines_config
from baselines.llm_client import LLMClient, UsageTracker
from baselines.prompt_context import build_context
from refinement_loop.adapters import GroqStructuredClient
from baselines.score import score_suite

comparison_function = input("Comparison function [function_25]: ").strip() or "function_25"
source_path = baselines_config.DATASET_DIR / f"{comparison_function}.py"
if not source_path.exists():
    raise FileNotFoundError(f"Dataset function not found: {source_path}")

ctx = build_context(source_path)

# Exactly the same Baseline-A prompt is used for both models.
def run_one_llm_comparison(provider):
    if provider == "gemini":
        tracker = UsageTracker()
        client = LLMClient(tracker=tracker)
    elif provider == "groq":
        tracker = UsageTracker()
        client = GroqStructuredClient(tracker=tracker, model=groq_model)
    else:
        raise ValueError(provider)

    result = baseline_a.run(ctx, client=client)
    score = score_suite(comparison_function, result.test_source)
    return result, score

gemini_result, gemini_score = run_one_llm_comparison("gemini")
groq_result, groq_score = run_one_llm_comparison("groq")

comparison_dir = baselines_config.LOGS_DIR / "llm_comparison"
comparison_dir.mkdir(parents=True, exist_ok=True)

def score_dict(result, score, provider, model):
    usage = result.tracker.summary()
    return {
        "provider": provider,
        "model": model,
        "function_id": comparison_function,
        "system_variant": "Baseline_A_one_call_head_to_head",
        "num_llm_calls": usage["num_llm_calls"],
        "input_tokens": usage["input_tokens"],
        "output_tokens": usage["output_tokens"],
        "thought_tokens": usage.get("thought_tokens", 0),
        "total_tokens_used": usage["total_tokens_used"],
        "estimated_cost_usd": usage["estimated_cost_usd"],
        "mutation_score_pct": round(score.mutation_score_pct, 2) if score else None,
        "line_coverage_pct": round(score.line_coverage_pct, 2) if score else None,
        "pass_rate_pct": round(score.pass_rate_pct, 2) if score else None,
        "num_generated_tests": len(result.suite.tests),
    }

gemini_record = score_dict(
    gemini_result, gemini_score, "gemini", baselines_config.MODEL_ID
)
groq_record = score_dict(
    groq_result, groq_score, "groq", groq_model
)

(comparison_dir / f"{comparison_function}__gemini.json").write_text(
    json.dumps(gemini_record, indent=2) + "\n", encoding="utf-8"
)
(comparison_dir / f"{comparison_function}__groq.json").write_text(
    json.dumps(groq_record, indent=2) + "\n", encoding="utf-8"
)

comparison_df = __import__("pandas").DataFrame([gemini_record, groq_record])
display(comparison_df[[
    "provider", "model", "function_id", "num_llm_calls",
    "num_generated_tests", "mutation_score_pct",
    "line_coverage_pct", "pass_rate_pct",
    "total_tokens_used", "estimated_cost_usd"
]])

print("\nDirect comparison completed: exactly one generation call per LLM.")
print("Saved under:", comparison_dir)


### How to interpret the two-LLM call

This head-to-head result is useful as a **direct model comparison point** because:
- the target function is identical,
- the prompt is identical,
- the requested output schema is identical,
- each model receives one generation call,
- both suites are measured by the same scorer.

Do not present this single-function result as the main statistical experiment. Use it as an illustrative cross-LLM comparison and then use the full repeated experiment for the paper's primary claims.


In [ ]:
# 6. Select the provider for the full 30-function experiment.
full_provider = input(
    "Provider for the full experiment [gemini/groq] (default: gemini): "
).strip().lower() or "gemini"

if full_provider not in {"gemini", "groq"}:
    raise ValueError("Provider must be 'gemini' or 'groq'.")

print("Full experiment provider:", full_provider)


In [ ]:
# 7. Cheapest live integration check for the selected full-experiment provider.
# This is separate from the two-LLM head-to-head call above.
function_id = comparison_function
variant = "error_trace"

cmd = [
    sys.executable, "-m", "refinement_loop.run_live",
    function_id, "--provider", full_provider, "--variant", variant
]
print("$", " ".join(cmd))
subprocess.run(cmd, check=True)
print("Selected-provider refinement smoke test passed.")


In [ ]:
# 8. Run Baseline A and Baseline B on the same smoke-test function.
# These baselines use the project's configured Gemini baseline client.
for module in ("baselines.baseline_a", "baselines.baseline_b"):
    cmd = [sys.executable, "-m", module, function_id, "--score"]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


## Full paper experiment

Run the following cells only after:
1. offline tests pass,
2. both LLM keys work,
3. the one-call Gemini/Groq comparison succeeds, and
4. the selected-provider refinement smoke test succeeds.

The full experiment keeps the existing locked design:
- 30 functions
- 3 repeats
- Baseline A
- Baseline B
- Proposed Variant 1
- Proposed Variant 2

Existing log + generated-test pairs are resumed/skipped unless `--force` is used.


In [ ]:
# 9. Baseline A: 30 functions × 3 repeats.
subprocess.run([
    sys.executable, "-m", "baselines.baseline_a",
    "--all", "--repeats", "3", "--score"
], check=True)


In [ ]:
# 10. Baseline B: 30 functions × 3 repeats.
subprocess.run([
    sys.executable, "-m", "baselines.baseline_b",
    "--all", "--repeats", "3", "--score"
], check=True)


In [ ]:
# 11. Proposed Variant 1: error-trace refinement.
subprocess.run([
    sys.executable, "-m", "refinement_loop.run_live",
    "--all", "--repeats", "3", "--provider", full_provider,
    "--variant", "error_trace"
], check=True)


In [ ]:
# 12. Proposed Variant 2: state-prediction refinement.
subprocess.run([
    sys.executable, "-m", "refinement_loop.run_live",
    "--all", "--repeats", "3", "--provider", full_provider,
    "--variant", "state_prediction"
], check=True)


In [ ]:
# 13. Strictly validate that the experiment produced logs before analysis.
from pathlib import Path
log_files = list(Path("logs").glob("*.json"))
if not log_files:
    raise RuntimeError("No experiment JSON logs found.")

print(f"Found {len(log_files)} JSON log files.")
print("The final analysis below reads only logs/*.json and ignores logs/llm_comparison/.")


## Analyze the logged experiment

The main tables below are generated from the actual JSON experiment logs. The separate `logs/llm_comparison/` records are deliberately excluded from the main repeated-experiment dataframe.


In [ ]:
# 14. Aggregate the main experiment logs and derive provider from filenames where available.
import glob, pandas as pd, numpy as np, re

records = []
for path in glob.glob("logs/*.json"):
    try:
        with open(path, encoding="utf-8") as f:
            row = json.load(f)
        row["log_file"] = path

        # Proposed logs include the provider in their filename.
        stem = Path(path).stem
        if stem.endswith("__gemini"):
            row["provider"] = "gemini"
        elif stem.endswith("__groq"):
            row["provider"] = "groq"
        elif row.get("system_variant") in {"Baseline_A", "Baseline_B"}:
            # Member-3 baseline client is Gemini in the finalized project.
            row["provider"] = "gemini"
        else:
            row["provider"] = row.get("provider", "unknown")

        records.append(row)
    except Exception as exc:
        print("Skipping unreadable log:", path, exc)

df = pd.DataFrame(records)
if df.empty:
    raise RuntimeError("No main experiment logs found.")

cols = [
    "function_id","system_variant","provider","iteration_count",
    "mutation_score_pct","line_coverage_pct","pass_rate_pct",
    "num_llm_calls","total_tokens_used","estimated_cost_usd"
]
for c in cols:
    if c not in df:
        df[c] = np.nan

summary_cols = [
    "mutation_score_pct","line_coverage_pct","pass_rate_pct",
    "num_llm_calls","total_tokens_used","estimated_cost_usd"
]

print("Main experiment records:", len(df))
display(
    df.groupby(["provider", "system_variant"])[summary_cols]
      .agg(["mean","std","count"])
      .round(3)
)


In [ ]:
# 15. Cross-LLM comparison table from the dedicated one-call records.
comparison_records = []
for path in sorted(Path("logs/llm_comparison").glob("*.json")):
    comparison_records.append(json.loads(path.read_text(encoding="utf-8")))

comparison_df = pd.DataFrame(comparison_records)
if not comparison_df.empty:
    display(comparison_df[[
        "provider","model","function_id","num_llm_calls",
        "mutation_score_pct","line_coverage_pct","pass_rate_pct",
        "total_tokens_used","estimated_cost_usd"
    ]])
else:
    print("No one-call comparison records found.")


In [ ]:
# 16. RQ1 paired comparison within the full experiment.
from scipy.stats import wilcoxon

def paired_scores(a, b, provider=None):
    subset = df if provider is None else df[df.provider == provider]
    x = subset[subset.system_variant == a].groupby("function_id").mutation_score_pct.mean()
    y = subset[subset.system_variant == b].groupby("function_id").mutation_score_pct.mean()
    joined = pd.concat([x, y], axis=1, keys=[a, b]).dropna()
    if len(joined) < 2:
        return joined, None
    return joined, wilcoxon(joined[a], joined[b], alternative="two-sided")

for provider_name in sorted(df.provider.dropna().unique()):
    for baseline in ["Baseline_A", "Baseline_B"]:
        joined, test = paired_scores("Variant_1_ErrorTrace", baseline, provider_name)
        print(f"\n{provider_name}: Variant_1 vs {baseline}; paired functions = {len(joined)}")
        if test is not None:
            print("Wilcoxon statistic:", test.statistic, "p-value:", test.pvalue)


In [ ]:
# 17. RQ2/RQ4: compare the two refinement prompting variants.
for provider_name in sorted(df.provider.dropna().unique()):
    joined, test = paired_scores(
        "Variant_1_ErrorTrace",
        "Variant_2_StatePrediction",
        provider_name
    )
    print(f"\n{provider_name}: Variant 1 vs Variant 2; paired functions = {len(joined)}")
    if test is not None:
        print("Wilcoxon statistic:", test.statistic, "p-value:", test.pvalue)


In [ ]:
# 18. RQ3: refinement iterations and cost/quality trade-off.
proposed = df[df.system_variant.isin([
    "Variant_1_ErrorTrace", "Variant_2_StatePrediction"
])].copy()

if not proposed.empty:
    display(
        proposed.groupby(["provider","system_variant","iteration_count"])
        .mutation_score_pct.mean()
        .round(2)
        .reset_index()
    )
    print("\nMean calls / tokens / cost:")
    display(
        proposed.groupby(["provider","system_variant"])[[
            "num_llm_calls","total_tokens_used","estimated_cost_usd"
        ]].mean().round(4)
    )


In [ ]:
# 19. Export paper-ready result files.
logs_dir = Path("logs")
main_summary = logs_dir / "final_experiment_summary.csv"
df.to_csv(main_summary, index=False)

if not comparison_df.empty:
    comparison_summary = logs_dir / "llm_comparison_summary.csv"
    comparison_df.to_csv(comparison_summary, index=False)
    print("Saved:", comparison_summary)

print("Saved:", main_summary)


## Reproducibility checklist

Before using numbers in the paper:

1. Offline acceptance tests passed.
2. Both Gemini and Groq keys were tested.
3. The **same single-call prompt** was sent once to each LLM for the direct comparison.
4. The two one-call suites were scored by the same mutation/coverage/pytest scorer.
5. The main experiment used the locked 30-function × 3-repeat design.
6. Do not mix `logs/llm_comparison/` with the main repeated-experiment statistics.
7. Report the actual provider/model, calls, tokens, cost, mutation score, coverage, and pass rate.
8. Use paired Wilcoxon tests on the same functions for system comparisons.
9. Do not use mock runs or smoke-test numbers as the paper's final experimental results.
10. Keep the raw JSON logs and CSV exports as the reproducibility artifacts.
